This notebook conducts a series of experiments on the effect of using different lower bouds for fitting our exceedance function, while comparing them to a fit on the original Marani et al. data.

In [15]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from ipywidgets import interactive
from scipy.stats import genpareto

In [16]:
novel_outbreaks = pd.read_excel("../../data/raw/novel_resp_241228.xlsx")

*Run `python/scipts/fit_quasi_original_marani.py` before loading distribution configs below*

In [17]:
with open("../../output/severity_distributions/genpareto_quasi_original_marani_sev.yaml", 'rb') as f:
  orig_marani_sev_config = yaml.safe_load(f)
  
with open("../../output/severity_distributions/genpareto_quasi_original_marani_int.yaml", 'rb') as f:
  orig_marani_int_config = yaml.safe_load(f)
  
with open("../../data/clean/inverted_covid_severity.yaml", 'rb') as f:
  inverted_covid_severity_dict = yaml.safe_load(f)
  inverted_covid_severity = inverted_covid_severity_dict['ex_ante_severity']

In [18]:
def plot_exceedance(lower_bound=1e-2, drop_hiv=False, use_inverted_covid=False):
    """
    Creates interactive plot comparing exceedance functions between original Marani data and novel outbreaks
    
    Args:
        lower_bound (float): Lower bound threshold for fitting Pareto distribution
        drop_hiv (bool): Whether to exclude HIV/AIDS observation
        use_inverted_covid (bool): Whether to use inverted COVID-19 severity
    """
    # Create copy of novel outbreaks data
    df = novel_outbreaks.copy()
    
    # Apply HIV filter if selected
    if drop_hiv:
        df = df[df['disease'] != 'hiv/aids']
        
    # Replace COVID severity/intensity if selected
    if use_inverted_covid:
        covid_mask = df['disease'] == 'covid-19'
        df.loc[covid_mask, 'severity_smu'] = inverted_covid_severity
    
    # Filter by lower bound
    df_sev = df[df['severity_smu'] >= lower_bound]
    
    df['intensity'] = df['severity_smu'] / df['duration']
    df_int = df[df['intensity'] >= lower_bound]
    
    # Fit Pareto distributions
    sev_shape, sev_loc, sev_scale = genpareto.fit(df_sev['severity_smu'].values, floc=lower_bound)
    int_shape, int_loc, int_scale = genpareto.fit(df_int['intensity'].values, floc=lower_bound)
    
    # Create figure with subplots
    fig = plt.figure(figsize=(15,8))
    gs = fig.add_gridspec(2, 2, height_ratios=[3, 1])
    ax1 = fig.add_subplot(gs[0,0])
    ax2 = fig.add_subplot(gs[0,1])
    text_ax1 = fig.add_subplot(gs[1,0])
    text_ax2 = fig.add_subplot(gs[1,1])
    
    # Generate points for theoretical curves
    x = np.logspace(np.log10(lower_bound), 3, 1000)
    
    # Plot original Marani fits
    y_sev_orig = 1 - genpareto.cdf(x, 
                                  c=orig_marani_sev_config['params']['k'],
                                  loc=orig_marani_sev_config['params']['theta'],
                                  scale=orig_marani_sev_config['params']['sigma'])
    
    y_int_orig = 1 - genpareto.cdf(x,
                                   c=orig_marani_int_config['params']['k'],
                                   loc=orig_marani_int_config['params']['theta'],
                                   scale=orig_marani_int_config['params']['sigma'])
    
    ax1.loglog(x, y_sev_orig, 'b-', label='~Original Marani Fit')
    ax2.loglog(x, y_int_orig, 'b-', label='~Original Marani Fit')
    
    # Plot new fits
    y_sev_new = 1 - genpareto.cdf(x, c=sev_shape, loc=sev_loc, scale=sev_scale)
    y_int_new = 1 - genpareto.cdf(x, c=int_shape, loc=int_loc, scale=int_scale)
    
    ax1.loglog(x, y_sev_new, 'r-', label='Novel Outbreaks Fit')
    ax2.loglog(x, y_int_new, 'r-', label='Novel Outbreaks Fit')
    
    # Plot empirical points and labels
    sev_sorted = np.sort(df_sev['severity_smu'].values)[::-1]
    int_sorted = np.sort(df_int['intensity'].values)[::-1]
    emp_exc_sev = np.arange(1, len(sev_sorted) + 1) / (len(sev_sorted) + 1)
    emp_exc_int = np.arange(1, len(int_sorted) + 1) / (len(int_sorted) + 1)
    
    ax1.loglog(sev_sorted, emp_exc_sev, 'ko', alpha=0.5)
    ax2.loglog(int_sorted, emp_exc_int, 'ko', alpha=0.5)
    
    # Add disease labels for severity plot
    for i, row in df_sev.iterrows():
        rank = (df_sev['severity_smu'] >= row['severity_smu']).sum()
        exc_prob = rank / (len(df_sev) + 1)
        ax1.annotate(row['disease'], (row['severity_smu'], exc_prob),
                    xytext=(5,5), textcoords='offset points', fontsize=8)
    
    # Add disease labels for intensity plot    
    for i, row in df_int.iterrows():
        rank = (df_int['intensity'] >= row['intensity']).sum()
        exc_prob = rank / (len(df_int) + 1)
        ax2.annotate(row['disease'], (row['intensity'], exc_prob),
                    xytext=(5,5), textcoords='offset points', fontsize=8)
    
    # Add vertical lines at lower bounds
    ax1.axvline(lower_bound, color='gray', linestyle='--', alpha=0.5)
    ax2.axvline(lower_bound, color='gray', linestyle='--', alpha=0.5)
    
    # Set labels and titles
    ax1.set_xlabel('Severity (deaths per 10k)')
    ax1.set_ylabel('Exceedance probability')
    ax1.set_title('Severity exceedance')
    ax1.legend()
    ax1.grid(True)
    
    ax2.set_xlabel('Intensity (deaths per 10k per year)')
    ax2.set_ylabel('Exceedance probability')
    ax2.set_title('Intensity exceedance')
    ax2.legend()
    ax2.grid(True)
    
    # Add parameter text displays
    text_ax1.axis('off')
    text_ax2.axis('off')
    
    sev_params = (
        f"Severity Parameters:\n"
        f"Original Marani:\n"
        f"  shape (k) = {orig_marani_sev_config['params']['k']:.3f}\n"
        f"  scale (σ) = {orig_marani_sev_config['params']['sigma']:.3f}\n"
        f"  loc   (θ) = {orig_marani_sev_config['params']['theta']:.3f}\n"
        f"Novel Outbreaks:\n"
        f"  k = {sev_shape:.3f}\n"
        f"  σ = {sev_scale:.3f}\n"
        f"  θ = {sev_loc:.3f}"
    )
    
    int_params = (
        f"Intensity Parameters:\n"
        f"Original Marani:\n"
        f"  shape (k) = {orig_marani_int_config['params']['k']:.3f}\n"
        f"  scale (σ) = {orig_marani_int_config['params']['sigma']:.3f}\n"
        f"  loc   (θ) = {orig_marani_int_config['params']['theta']:.3f}\n"
        f"Novel Outbreaks:\n"
        f"  k = {int_shape:.3f}\n"
        f"  σ = {int_scale:.3f}\n"
        f"  θ = {int_loc:.3f}"
    )
    
    text_ax1.text(0.1, 0.5, sev_params, fontfamily='monospace')
    text_ax2.text(0.1, 0.5, int_params, fontfamily='monospace')
    
    plt.tight_layout()
    return fig

In [19]:
# Create interactive widget
interactive_plot = interactive(
    plot_exceedance,
    lower_bound=widgets.FloatLogSlider(value=1e-2, min=-6, max=1, step=0.1, description = 'Lower bound'),
    drop_hiv=widgets.Checkbox(value=False, description='Drop HIV/AIDS'),
    use_inverted_covid=widgets.Checkbox(value=False, description='Use Inverted COVID')
)

display(interactive_plot)

interactive(children=(FloatLogSlider(value=0.01, description='Lower bound', max=1.0, min=-6.0), Checkbox(value…

### Do more general parameter exploration

In [14]:
def plot_gpd_exceedance(shape, loc, scale):
    """
    Plot the exceedance function of a Generalized Pareto Distribution with given parameters.
    
    Args:
        shape (float): Shape parameter (k) of the GPD
        loc (float): Location parameter (θ) of the GPD  
        scale (float): Scale parameter (σ) of the GPD
    
    Returns:
        matplotlib.figure.Figure: Figure containing the exceedance plot
    """
    fig, ax = plt.subplots(figsize=(10, 6))
    
    # Generate x values
    x_max = 100
    x = np.linspace(loc, x_max, 1000)
    
    # Calculate exceedance probabilities
    # P(X > x) = (1 + shape*(x-loc)/scale)^(-1/shape)
    exceedance = 1 - genpareto.cdf(x, shape, loc=loc, scale=scale)
    
    # Plot on log-y scale
    ax.loglog(x, exceedance)
    ax.grid(True)
    ax.set_xlabel('x')
    ax.set_ylabel('P(X > x)')
    ax.set_title('GPD Exceedance Function')
    
    # Set fixed axis limits
    ax.set_xlim(loc, x_max)
    ax.set_ylim(1e-3, 1)
    
    # Add parameter text
    param_text = f'k (shape) = {shape:.2f}\nθ (loc) = {loc:.2f}\nσ (scale) = {scale:.2f}'
    ax.text(0.95, 0.95, param_text,
            transform=ax.transAxes,
            verticalalignment='top',
            horizontalalignment='right',
            bbox=dict(facecolor='white', alpha=0.8))
    
    return fig

# Create interactive widget
interactive_gpd = interactive(
    plot_gpd_exceedance,
    shape=widgets.FloatSlider(value=0.1, min=-1, max=2, step=0.1, description='Shape (k)'),
    loc=widgets.FloatSlider(value=0, min=-2, max=2, step=0.1, description='Location (θ)'),
    scale=widgets.FloatSlider(value=1, min=0.1, max=5, step=0.1, description='Scale (σ)')
)

display(interactive_gpd)


interactive(children=(FloatSlider(value=0.1, description='Shape (k)', max=2.0, min=-1.0), FloatSlider(value=0.…